# Roasting a batch

Every other notebook here simulates something the project made up. This one simulates a
machine that exists: a **Loring S15**, fifteen kilograms of green coffee at a time, which
roasted five batches on the evening of 30 July 2026 while a bridge logged it over Modbus
every one and a half seconds.

So there is a right answer, and it was not computed — it was measured. The figures below
put the model's curve on top of the roaster's own bean probe and leave the gap visible.

This notebook is meant to be **run**. The figures are drawn by matplotlib when the site is
built; the slider at the end needs a live kernel, so on this page it is a snapshot of its
last state. The file is attached — [roasting.ipynb](roasting.ipynb) — but it reads the
model from `examples/`, so it wants a checkout rather than a bare download:

```bash
git clone https://gitlab.com/jorgeecardona/pymodelica && cd pymodelica
pip install "modelica[sim]" matplotlib ipywidgets
jupyter lab docs/notebooks/roasting.ipynb
```

The measurement it checks against is carried inline, a little further down. The roast logs
live in a different repository and this page does not want to depend on one.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# The model comes from `examples/` in the repository rather than from a string in here: a copy
# pasted into a notebook is a copy that goes stale, and the reason this page is executed when the
# site is built is that it then cannot show output from a model the project does not have.
root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "models").is_dir()), None
)
if root is None:
    message = (
        "This notebook reads its model from `examples/` in the pymodelica repository. "
        "Run it from a checkout:\n"
        "    git clone https://gitlab.com/jorgeecardona/pymodelica\n"
        "    cd pymodelica && jupyter lab docs/notebooks/"
    )
    raise RuntimeError(message)

SURFACE, INK, MUTED, LINE = "#fcfcfb", "#0b0b0b", "#52514e", "#d4d3cd"
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"

mpl.rcParams.update(
    {
        "figure.figsize": (8.4, 3.4),
        "figure.dpi": 110,
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": LINE,
        "axes.labelcolor": MUTED,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.titlelocation": "left",
        "axes.titlepad": 10,
        "axes.titlesize": 11,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "lines.linewidth": 2.0,
        "legend.frameon": False,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "font.size": 10,
    }
)
%config InlineBackend.figure_formats = ["svg"]


def minutes(ax):
    """Roasts are read in minutes and seconds, not in seconds."""
    ax.set_xlabel("time from charge (min)")
    ax.xaxis.set_major_formatter(lambda s, _: f"{int(s // 60)}:{int(s % 60):02d}")
    ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(120))


def label(ax, x, y, text, colour, dy=0):
    """Direct labels. Four series here sit under 3:1 against the surface, so every one of
    them is named on the plot rather than left to colour alone."""
    ax.annotate(
        text,
        xy=(x, y),
        xytext=(6, dy),
        textcoords="offset points",
        color=colour,
        fontsize=9,
        va="center",
    )

## The machine, as eighteen lines

Two lumps of heat and a gas valve. `T_mach` is everything hot that is not coffee — steel,
lining, and the air going round the recirculation loop; `T_bean` is the charge. The burner
pushes heat into the first, the first leaks some to the room and hands the rest to the
second.

The nine-level chain on `burner` is not invented. It is this batch's own recorded gas
trace, reduced to the steps worth carrying.

In [ ]:
print((root / "examples" / "models" / "RoastingMachine.mo").read_text())

## One batch

`parse` → `flatten` → `causalize` → `simulate`, the same four calls as every other model in
the corpus. What comes back is fourteen and a half minutes of roast.

In [ ]:
from modelica.lang import parse
from modelica.sim import causalize, flatten, simulate

flat = flatten([parse((root / "examples" / "models" / "RoastingMachine.mo").read_text())], "RoastingMachine")
system = causalize(flat)

STOP = 863.6
result = simulate(system, stop=STOP, tolerance=1e-10, points=900)

print(f"states     {sorted(system.states)}")
print(f"events     {len(result.events)}")
print(f"drop       {result.at(STOP)['bean']:.1f} degC at {STOP / 60:.2f} min")

## Where the clock makes the events

Nothing here crosses a threshold. `DrainingTank` fires when the level reaches zero and
`BouncingBall` when the ball reaches the floor, but a gas schedule fires because it is
twenty past ten — the condition is on `time` itself, and the solver knows every one of
these instants before it starts integrating.

Eight steps in the gas, eight events. The burner is a separate panel rather than a second
y-axis: per cent and degrees are different measures and do not belong on one scale.

In [ ]:
for event in result.events:
    print(f"  t = {event.time:6.1f} s  ({event.time / 60:5.2f} min)   {event.cause}")

In [ ]:
t = result.time
bean = result.values[result.names.index("bean")]
mach = result.values[result.names.index("T_mach")] - 273.15
gas = result.values[result.names.index("burner")]

fig, (top, bottom) = plt.subplots(
    2, 1, figsize=(8.4, 4.6), sharex=True, height_ratios=[2.4, 1], gridspec_kw={"hspace": 0.15}
)

top.plot(t, mach, color=AQUA)
top.plot(t, bean, color=BLUE)
label(top, t[-1], mach[-1], "machine", AQUA, dy=7)
label(top, t[-1], bean[-1], "beans", BLUE, dy=-7)
top.set_title("The two lumps, and the gas that drives them")
top.set_ylabel("degC")
top.set_xlim(0, STOP * 1.09)

bottom.plot(t, gas, color=YELLOW, drawstyle="steps-post")
label(bottom, t[-1], gas[-1], "burner", YELLOW)
bottom.set_ylabel("%")
bottom.set_ylim(0, 100)
minutes(bottom)

for event in result.events:
    for ax in (top, bottom):
        ax.axvline(event.time, color=LINE, linewidth=0.8, zorder=0)

fig.legend(
    handles=[
        mpl.lines.Line2D([], [], color=AQUA, label="machine temperature"),
        mpl.lines.Line2D([], [], color=BLUE, label="bean temperature"),
        mpl.lines.Line2D([], [], color=YELLOW, label="burner"),
        mpl.lines.Line2D([], [], color=LINE, linewidth=0.8, label="time event"),
    ],
    loc="lower center",
    ncol=4,
    bbox_to_anchor=(0.5, -0.06),
)
plt.show()

## What the roaster actually read

Here is the measurement. Five batches, downsampled to fifteen-second spacing and carried
inline so this page owes nothing to another repository: for each one, the gas trace the
machine followed and what its bean probe reported.

In [ ]:
BATCHES = [
    # batch 1: 14.39 min, 9 gas steps
    {
        "stop": 863.63,
        "turn": 46.75,
        "gas": [(0.0, 20.0), (66.7, 27.01), (93.0, 84.97), (526.9, 79.29), (618.0, 70.0), (688.5, 61.46), (714.6, 50.0), (754.4, 37.42), (780.8, 20.0)],
        "bt": [
            (0.0, 216.83), (15.0, 102.84), (30.0, 70.45), (45.0, 64.49), (60.0, 67.25),
            (75.0, 72.73), (90.0, 79.94), (105.0, 87.57), (120.0, 94.87), (135.0, 101.41),
            (150.0, 107.61), (165.0, 113.39), (180.0, 118.96), (195.0, 123.84),
            (210.0, 127.71), (225.0, 131.7), (240.0, 135.39), (255.0, 138.73), (270.0, 141.76),
            (285.0, 144.94), (300.0, 147.65), (315.0, 150.33), (330.0, 153.13),
            (345.0, 155.73), (360.0, 158.27), (375.0, 161.0), (390.0, 163.16), (405.0, 165.6),
            (420.0, 168.11), (435.0, 170.18), (450.0, 172.4), (465.0, 174.37), (480.0, 176.53),
            (495.0, 178.58), (510.0, 180.56), (525.0, 182.76), (540.0, 184.74),
            (555.0, 186.74), (570.0, 188.74), (585.0, 190.57), (600.0, 192.48),
            (615.0, 194.79), (630.0, 196.86), (645.0, 198.89), (660.0, 201.03),
            (675.0, 203.12), (690.0, 205.32), (705.0, 207.26), (720.0, 208.83),
            (735.0, 210.03), (750.0, 211.07), (765.0, 211.94), (780.0, 212.28),
            (795.0, 212.68), (810.0, 213.5), (825.0, 214.15), (840.0, 214.68), (855.0, 215.56),
            (863.6, 216.06)
        ],
    },
    # batch 2: 13.90 min, 28 gas steps
    {
        "stop": 833.7,
        "turn": 43.83,
        "gas": [(0.0, 20.0), (50.0, 74.5), (76.2, 27.19), (102.2, 50.99), (128.1, 100.0), (154.2, 78.47), (190.9, 84.56), (227.6, 90.58), (256.8, 98.53), (283.0, 92.51), (308.8, 100.0), (334.7, 93.85), (360.8, 99.59), (387.0, 93.21), (413.1, 100.0), (486.5, 94.91), (512.8, 100.0), (538.9, 91.92), (565.2, 78.18), (591.3, 64.21), (623.3, 54.44), (654.0, 49.06), (680.3, 38.88), (706.3, 67.48), (732.5, 55.43), (759.5, 50.11), (785.2, 37.54), (811.0, 20.0)],
        "bt": [
            (0.0, 192.17), (15.0, 100.61), (30.0, 68.4), (45.0, 62.76), (60.0, 66.1),
            (75.0, 73.07), (90.0, 80.12), (105.0, 86.27), (120.0, 92.48), (135.0, 99.1),
            (150.0, 105.64), (165.0, 111.43), (180.0, 116.5), (195.0, 121.2), (210.0, 125.92),
            (225.0, 129.94), (240.0, 133.96), (255.0, 137.77), (270.0, 141.22),
            (285.0, 144.62), (300.0, 147.44), (315.0, 150.52), (330.0, 153.85), (345.0, 156.5),
            (360.0, 159.14), (375.0, 161.82), (390.0, 164.49), (405.0, 167.17),
            (420.0, 169.58), (435.0, 172.13), (450.0, 174.57), (465.0, 177.0), (480.0, 179.52),
            (495.0, 181.95), (510.0, 184.26), (525.0, 186.59), (540.0, 189.03),
            (555.0, 191.16), (570.0, 193.3), (585.0, 195.54), (600.0, 197.32), (615.0, 198.64),
            (630.0, 200.58), (645.0, 202.18), (660.0, 203.49), (675.0, 204.98),
            (690.0, 206.33), (705.0, 207.51), (720.0, 209.04), (735.0, 210.66),
            (750.0, 211.87), (765.0, 212.57), (780.0, 213.68), (795.0, 214.65), (810.0, 215.5),
            (825.0, 216.35), (833.7, 216.94)
        ],
    },
    # batch 3: 14.27 min, 24 gas steps
    {
        "stop": 856.39,
        "turn": 44.26,
        "gas": [(0.0, 20.0), (50.3, 87.48), (75.8, 20.0), (101.3, 30.99), (127.0, 100.0), (152.6, 92.63), (178.2, 79.76), (205.3, 69.06), (230.9, 100.0), (273.0, 92.63), (301.7, 99.23), (429.5, 91.52), (455.1, 100.0), (504.6, 93.56), (530.1, 99.29), (555.8, 74.15), (581.3, 79.76), (606.9, 53.97), (632.6, 44.26), (658.3, 51.46), (683.7, 43.97), (709.4, 60.76), (735.1, 34.97), (786.8, 22.45)],
        "bt": [
            (0.0, 171.56), (15.0, 90.21), (30.0, 64.74), (45.0, 61.55), (60.0, 65.8),
            (75.0, 73.38), (90.0, 80.39), (105.0, 86.41), (120.0, 92.25), (135.0, 98.54),
            (150.0, 105.35), (165.0, 111.05), (180.0, 116.35), (195.0, 121.25),
            (210.0, 125.76), (225.0, 129.51), (240.0, 133.19), (255.0, 137.1), (270.0, 140.84),
            (285.0, 144.41), (300.0, 147.29), (315.0, 150.17), (330.0, 153.32),
            (345.0, 156.22), (360.0, 158.83), (375.0, 161.53), (390.0, 164.25), (405.0, 166.9),
            (420.0, 169.52), (435.0, 172.23), (450.0, 174.46), (465.0, 176.73),
            (480.0, 179.22), (495.0, 181.66), (510.0, 184.04), (525.0, 186.28),
            (540.0, 188.73), (555.0, 191.02), (570.0, 192.94), (585.0, 195.19),
            (600.0, 196.97), (615.0, 198.83), (630.0, 200.38), (645.0, 201.69), (660.0, 203.4),
            (675.0, 204.85), (690.0, 206.14), (705.0, 207.46), (720.0, 208.56),
            (735.0, 209.44), (750.0, 210.11), (765.0, 210.82), (780.0, 211.61),
            (795.0, 212.41), (810.0, 212.96), (825.0, 213.83), (840.0, 214.58),
            (855.0, 215.47), (856.4, 215.72)
        ],
    },
    # batch 4: 14.18 min, 21 gas steps
    {
        "stop": 851.0,
        "turn": 42.26,
        "gas": [(0.0, 20.0), (49.7, 100.0), (75.4, 20.0), (101.0, 43.74), (126.6, 100.0), (152.1, 87.89), (177.7, 74.21), (203.3, 89.23), (228.7, 97.07), (344.5, 90.7), (370.0, 96.37), (539.3, 89.12), (565.0, 79.88), (592.2, 73.09), (617.9, 44.03), (643.6, 50.58), (669.4, 35.55), (695.3, 63.8), (720.9, 50.0), (746.7, 30.0), (776.9, 20.0)],
        "bt": [
            (0.0, 177.67), (15.0, 91.93), (30.0, 65.63), (45.0, 62.07), (60.0, 66.05),
            (75.0, 73.85), (90.0, 80.54), (105.0, 86.71), (120.0, 92.65), (135.0, 99.39),
            (150.0, 105.76), (165.0, 111.53), (180.0, 116.95), (195.0, 121.48),
            (210.0, 126.03), (225.0, 130.09), (240.0, 133.98), (255.0, 137.62), (270.0, 141.2),
            (285.0, 144.54), (300.0, 147.69), (315.0, 150.67), (330.0, 153.66),
            (345.0, 156.75), (360.0, 159.4), (375.0, 162.06), (390.0, 164.67), (405.0, 167.23),
            (420.0, 169.76), (435.0, 172.28), (450.0, 174.93), (465.0, 177.29),
            (480.0, 179.65), (495.0, 181.88), (510.0, 184.3), (525.0, 186.78), (540.0, 189.24),
            (555.0, 191.47), (570.0, 193.45), (585.0, 195.48), (600.0, 197.37),
            (615.0, 199.24), (630.0, 200.62), (645.0, 202.2), (660.0, 203.73), (675.0, 205.04),
            (690.0, 206.3), (705.0, 207.66), (720.0, 208.93), (735.0, 210.06), (750.0, 211.06),
            (765.0, 211.84), (780.0, 212.44), (795.0, 212.94), (810.0, 213.7), (825.0, 214.42),
            (840.0, 215.32), (851.0, 215.89)
        ],
    },
    # batch 5: 13.98 min, 10 gas steps
    {
        "stop": 838.71,
        "turn": 40.97,
        "gas": [(0.0, 20.0), (60.8, 25.96), (86.6, 90.0), (564.4, 84.97), (620.9, 70.0), (684.6, 63.85), (710.3, 50.0), (768.1, 42.51), (794.0, 34.97), (820.5, 20.0)],
        "bt": [
            (0.0, 179.5), (15.0, 91.11), (30.0, 65.98), (45.0, 62.81), (60.0, 66.53),
            (75.0, 72.43), (90.0, 80.0), (105.0, 87.43), (120.0, 94.44), (135.0, 101.1),
            (150.0, 106.88), (165.0, 112.48), (180.0, 117.62), (195.0, 122.45),
            (210.0, 126.76), (225.0, 130.85), (240.0, 134.56), (255.0, 138.03),
            (270.0, 141.31), (285.0, 144.22), (300.0, 147.36), (315.0, 150.19), (330.0, 153.1),
            (345.0, 156.08), (360.0, 158.65), (375.0, 161.15), (390.0, 163.95),
            (405.0, 166.43), (420.0, 168.89), (435.0, 171.46), (450.0, 173.67),
            (465.0, 175.98), (480.0, 178.22), (495.0, 180.24), (510.0, 182.14), (525.0, 184.6),
            (540.0, 186.66), (555.0, 188.87), (570.0, 191.04), (585.0, 193.11),
            (600.0, 195.21), (615.0, 197.37), (630.0, 199.72), (645.0, 201.54),
            (660.0, 203.38), (675.0, 205.2), (690.0, 206.67), (705.0, 207.69), (720.0, 208.52),
            (735.0, 209.24), (750.0, 209.85), (765.0, 210.72), (780.0, 211.59),
            (795.0, 212.48), (810.0, 213.45), (825.0, 214.28), (838.7, 215.11)
        ],
    },
]

# total points: 290


print(f"{len(BATCHES)} batches, {sum(len(b['bt']) for b in BATCHES)} points")
for n, b in enumerate(BATCHES, 1):
    print(
        f"  batch {n}: {b['stop'] / 60:5.2f} min, {len(b['gas'])} gas steps, "
        f"turn {b['turn']:4.1f} s, drop {b['bt'][-1][1]:6.2f} degC"
    )

### The first ninety seconds are the probe, not the beans

One thing to get out of the way before the curves go on top of each other. At charge the
bean probe is reading the empty drum it has been sitting in — 216 °C — and then fifteen
kilos of room-temperature coffee bury it. What the log shows over the next forty-odd
seconds is a thermocouple travelling from one temperature to another, bottoming out near
62 °C at what roasters call the *turning point*.

That is an instrument in transit. The beans were never at 216 °C and never at 62 °C. The
model starts them at ambient and integrates, which is right about the coffee and says
nothing about the probe, so nothing before 90 s is compared.

In [ ]:
b1 = BATCHES[0]
m_t = np.array([p[0] for p in b1["bt"]])
m_bt = np.array([p[1] for p in b1["bt"]])

SETTLED = 90.0
model = np.array([result.at(float(x))["bean"] for x in m_t])
after = m_t >= SETTLED
residual = model[after] - m_bt[after]

fig, (top, bottom) = plt.subplots(
    2, 1, figsize=(8.4, 4.6), sharex=True, height_ratios=[2.4, 1], gridspec_kw={"hspace": 0.15}
)

top.axvspan(0, SETTLED, color="#f2f1ec", zorder=0)
top.plot(m_t, m_bt, color=ORANGE)
top.plot(t, bean, color=BLUE)
label(top, m_t[-1], m_bt[-1], "measured", ORANGE, dy=9)
label(top, t[-1], bean[-1], "model", BLUE, dy=-9)
top.annotate(
    "probe leaving the drum\n(not compared)",
    xy=(46, 64),
    xytext=(120, 40),
    fontsize=8.5,
    color=MUTED,
    arrowprops={"arrowstyle": "-", "color": MUTED, "linewidth": 0.8},
)
top.set_title("Batch 1: the model against the machine")
top.set_ylabel("degC")
top.set_xlim(0, STOP * 1.09)

bottom.axhline(0, color=LINE, linewidth=0.8)
bottom.plot(m_t[after], residual, color=BLUE)
bottom.set_ylabel("model - measured (K)")
bottom.set_ylim(-5, 5)
minutes(bottom)

fig.legend(
    handles=[
        mpl.lines.Line2D([], [], color=ORANGE, label="measured (bean probe)"),
        mpl.lines.Line2D([], [], color=BLUE, label="model"),
    ],
    loc="lower center",
    ncol=2,
    bbox_to_anchor=(0.5, -0.06),
)
plt.show()

print(f"from {SETTLED:.0f} s to the drop, over {after.sum()} samples:")
print(f"  rms       {np.sqrt((residual**2).mean()):.2f} K")
print(f"  worst     {np.abs(residual).max():.2f} K")
print(f"  at drop   model {model[-1]:.1f} degC, measured {m_bt[-1]:.1f} degC")

## Rate of rise

Nobody roasting coffee watches the temperature. They watch its slope — *rate of rise*, in
degrees per minute — because that is the thing that has to come down smoothly. A roast
whose RoR crosses back upward has usually just been ruined, and no one can see that in the
temperature curve.

It is a derivative of a measurement, so it is noisy where the measurement is quantised.
The model's is not noisy at all, which is the honest difference between an integrated
curve and a thermocouple.

In [ ]:
def rate_of_rise(times, values, after=0.0):
    """degC per minute, and the instants it is defined at.

    An event puts two samples at the same instant -- the value on each side of the
    discontinuity, which is the whole point of restarting there -- and a centred difference
    over a zero-width interval is a division by zero. So the duplicates come out first.
    """
    keep = np.concatenate(([True], np.diff(times) > 0)) & (times >= after)
    kept, kept_values = times[keep], values[keep]
    return kept, np.gradient(kept_values, kept) * 60.0


fig, ax = plt.subplots(figsize=(8.4, 3.4))

m_ror_t, m_ror = rate_of_rise(m_t, m_bt, after=SETTLED)
ror_t, ror = rate_of_rise(t, bean, after=SETTLED)
ax.plot(m_ror_t, m_ror, color=ORANGE, linewidth=1.4, alpha=0.9)
ax.plot(ror_t, ror, color=BLUE)
label(ax, m_ror_t[-1], m_ror[-1], "measured", ORANGE, dy=9)
label(ax, ror_t[-1], ror[-1], "model", BLUE, dy=-9)

ax.set_title("Rate of rise: what a roaster is actually watching")
ax.set_ylabel("degC / min")
ax.set_ylim(0, None)
ax.set_xlim(0, STOP * 1.09)
minutes(ax)
plt.show()

## One parameter set, five batches

The parameters were fitted to batch 1 and to nothing else. The other four are the actual
test: same numbers, each batch's own gas trace, no refitting. They are not identical
roasts — the gas was worked differently on each, from nine steps on the first to
twenty-eight on the second.

Where it drifts, it drifts *hot*, and increasingly so down the session. That is the shape
you would expect from a machine carrying a little more heat into each batch than one fitted
initial condition can know about.

In [ ]:
from dataclasses import replace

from modelica.ir import nodes as ir
from modelica.ir.flat import FlatModel


def with_gas(model: FlatModel, steps) -> FlatModel:
    """The same model with a different gas schedule.

    The schedule lives in the equation for `burner` as a chain of comparisons against
    `time`, so swapping it is a matter of rebuilding that chain -- which is exactly what
    the Python front-end does when it folds a table, and what `unparse` prints back as
    `if ... elseif ... else`.
    """
    # Level `i` holds from steps[i][0] until steps[i+1][0], so each comparison carries the
    # level *before* it: `if time < steps[1].at then steps[0].level elseif ...`, ending on
    # the last level, which has nothing after it to stop it.
    chain: ir.Expr = ir.RealLiteral(float(steps[-1][1]))
    for index in range(len(steps) - 1, 0, -1):
        chain = ir.IfExpr(
            ir.Binary("<", ir.Ref(("time",)), ir.RealLiteral(float(steps[index][0]))),
            ir.RealLiteral(float(steps[index - 1][1])),
            chain,
        )
    return replace(
        model,
        equations=tuple(
            replace(eq, rhs=chain)
            if isinstance(eq.lhs, ir.Ref) and eq.lhs.parts == ("burner",)
            else eq
            for eq in model.equations
        ),
    )


def roast(batch) -> object:
    """Run the model against one recorded batch, on that batch's own gas."""
    return simulate(
        causalize(with_gas(flat, batch["gas"])),
        stop=batch["stop"],
        tolerance=1e-10,
        points=700,
    )


runs = [roast(b) for b in BATCHES]

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(9.6, 2.9), sharex=True, sharey=True)

print(f"{'batch':>6} {'rms':>7} {'worst':>7} {'drop measured':>15} {'model':>8}")
for n, (ax, b, run) in enumerate(zip(axes, BATCHES, runs, strict=True), 1):
    bt_t = np.array([p[0] for p in b["bt"]])
    bt_v = np.array([p[1] for p in b["bt"]])
    keep = bt_t >= SETTLED
    sim = np.array([run.at(float(x))["bean"] for x in bt_t])
    err = sim[keep] - bt_v[keep]

    ax.plot(bt_t[keep], bt_v[keep], color=ORANGE, linewidth=1.6)
    ax.plot(run.time, run.values[run.names.index("bean")], color=BLUE, linewidth=1.6)
    ax.set_title(f"batch {n}", fontsize=10)
    ax.set_xticks([0, 300, 600, 900])
    ax.xaxis.set_major_formatter(lambda s, _: f"{int(s // 60)}")
    ax.set_xlabel("min")
    if n == 1:
        ax.set_ylabel("degC")

    print(
        f"{n:>6} {np.sqrt((err**2).mean()):6.2f} K {np.abs(err).max():6.2f} K "
        f"{bt_v[-1]:14.1f} {sim[-1]:8.1f}"
    )

axes[0].legend(
    handles=[
        mpl.lines.Line2D([], [], color=ORANGE, label="measured"),
        mpl.lines.Line2D([], [], color=BLUE, label="model"),
    ],
    loc="lower right",
    fontsize=8,
)
fig.tight_layout()
fig.subplots_adjust(top=0.78)
fig.suptitle(
    "Fitted on batch 1, run unchanged on all five", x=0.01, y=0.98, ha="left", fontsize=11
)
plt.show()

## The knobs a roaster actually has

Three of them, and all three are parameters of the flat model, so tuning them is
`dataclasses.replace` and a re-causalization — the same trick as
[Turning the knobs](exploring.ipynb).

* **charge** scales `C_bean`: less coffee, same machine, so it climbs faster.
* **preheat** is where the machine starts, which is the number the roaster sets before the
  first batch and the one the session drifts away from by batch five.
* **gas trim** scales `Q_max`, standing in for a different line pressure or a dirtier
  burner.

The readout is the pair a roaster cares about: how hot it was when it came out, and how
long it took to get to 196 °C, which is about where first crack starts.

In [ ]:
def tuned(model: FlatModel, values: dict, preheat: float | None = None) -> FlatModel:
    """A copy with those parameters bound to those numbers, and optionally a new preheat."""
    return replace(
        model,
        variables=tuple(
            replace(
                v,
                binding=ir.RealLiteral(float(values[v.name])) if v.name in values else v.binding,
                start=(
                    ir.RealLiteral(preheat + 273.15)
                    if preheat is not None and v.name == "T_mach"
                    else v.start
                ),
            )
            for v in model.variables
        ),
    )


def crack(run, threshold: float = 196.0) -> float | None:
    """When the beans first reach `threshold`, interpolated between output points."""
    times = run.time
    bean = run.values[run.names.index("bean")]
    hit = np.flatnonzero(bean >= threshold)
    if not len(hit):
        return None
    k = hit[0]
    if k == 0:
        return float(times[0])
    lo, hi = bean[k - 1], bean[k]
    return float(times[k - 1] + (threshold - lo) / (hi - lo) * (times[k] - times[k - 1]))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

BASE_CHARGE = 15.0
BASE_C_BEAN = 3.0e4
BASE_Q_MAX = 1.25e5


def show(charge: float, preheat: float, trim: float) -> None:
    run = simulate(
        causalize(
            tuned(
                flat,
                {
                    "C_bean": BASE_C_BEAN * charge / BASE_CHARGE,
                    "Q_max": BASE_Q_MAX * trim,
                },
                preheat=preheat,
            )
        ),
        stop=STOP,
        tolerance=1e-9,
        points=700,
    )
    fc = crack(run)

    fig, ax = plt.subplots(figsize=(8.4, 3.4))
    ax.plot(m_t[after], m_bt[after], color=ORANGE, linewidth=1.4)
    ax.plot(run.time, run.values[run.names.index("bean")], color=BLUE)
    label(ax, m_t[-1], m_bt[-1], "as roasted", ORANGE, dy=9)
    label(ax, run.time[-1], run.values[run.names.index("bean")][-1], "this batch", BLUE, dy=-9)
    if fc is not None:
        ax.plot([fc], [196.0], marker="o", markersize=8, color=BLUE, zorder=5)
        ax.axhline(196.0, color=LINE, linewidth=0.8, zorder=0)

    drop = run.at(STOP)["bean"]
    crack_text = f"{fc / 60:.2f} min" if fc is not None else "never"
    ax.set_title(f"drop {drop:.1f} degC   ·   196 degC at {crack_text}")
    ax.set_ylabel("degC")
    ax.set_ylim(0, 260)
    ax.set_xlim(0, STOP * 1.09)
    minutes(ax)
    ax.legend(
        handles=[
            mpl.lines.Line2D([], [], color=ORANGE, label="batch 1, as roasted"),
            mpl.lines.Line2D([], [], color=BLUE, label="model, with these knobs"),
        ],
        loc="lower right",
    )
    plt.show()


widgets.interact(
    show,
    charge=widgets.FloatSlider(value=15.0, min=8.0, max=18.0, step=0.5, description="charge (kg)"),
    preheat=widgets.FloatSlider(
        value=151.3, min=120.0, max=190.0, step=1.0, description="preheat (degC)"
    ),
    trim=widgets.FloatSlider(value=1.0, min=0.7, max=1.3, step=0.02, description="gas trim"),
)

## What this model cannot tell you

Roasting drives about a tenth of the charge off as steam, and boiling that water off is a
large part of where the gas goes in the first half. There is no water in this model.

Not an oversight — the session recorded no weight in, no weight out and no moisture, and
without one of those there is nothing to calibrate a drying term against. An evaporation
term with invented parameters would have made the curve fit just as well and meant less,
so the latent heat is folded into `C_bean` instead. What makes that defensible rather than
a fudge is where the number lands: fitted freely, `C_bean` comes out at **2006 J/(kg·K)**
over a 15 kg charge, and dry coffee is about 1800. The excess is roughly the missing
drying, sitting where it belongs.

It also means `C_bean` is not really a heat capacity, so the charge slider above is lying
slightly: it scales the drying along with the coffee, which is right in direction and
approximate in size.

One weighing per batch — green in, roasted out — would be enough to split those apart.
Until then the model is honest about being a two-lump fit to a curve, and
[`RoastingMachine.toml`](https://gitlab.com/jorgeecardona/pymodelica/-/blob/main/examples/models/RoastingMachine.toml)
says so at length, alongside the measured points this notebook has been drawing.